# Combined Results — Figures + Comparison

**Run this AFTER both `colab_llama.ipynb` and `colab_gemma.ipynb` have finished.**

This notebook:
1. Clones the repo
2. Lets you upload the result zips from both model sessions
3. Generates all figures (LLaMA vs Gemma comparison plots)
4. Shows a side-by-side summary table
5. Downloads the figures zip

**No GPU needed** — figures are generated from JSON files, no model loading.

## Setup

In [ ]:
import os, sys

GITHUB_URL = 'https://github.com/AliHasan-786/llm-invisible-watermarking.git'
BRANCH     = 'AmmarChanges'
REPO_DIR   = 'llm-invisible-watermarking'

if not os.path.exists(REPO_DIR):
    print(f'Cloning {BRANCH}...')
    os.system(f'git clone --branch {BRANCH} {GITHUB_URL} {REPO_DIR}')
else:
    print('Pulling latest...')
    os.system(f'cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull')

os.chdir(REPO_DIR)
sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')

os.system('pip install -q matplotlib numpy scipy')
os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)
print('✓ Ready')

---
## Step 1 — Upload results from both model sessions

Run the two cells below. Each opens a file picker — select the zip you downloaded from the LLaMA session, then the one from the Gemma session.

The zips will be extracted into `results/` so all JSON files are in the right place for figure generation.

In [ ]:
import zipfile
from google.colab import files

print('Select the LLaMA results zip (results_llama_*.zip):')
uploaded = files.upload()
for fname, data in uploaded.items():
    with open(f'/content/{fname}', 'wb') as f:
        f.write(data)
    with zipfile.ZipFile(f'/content/{fname}') as zf:
        zf.extractall('.')
    print(f'✓ Extracted {fname}')

import os
llama_files = [f for f in os.listdir('results') if 'llama' in f]
print(f'LLaMA result files: {llama_files}')

In [ ]:
import zipfile
from google.colab import files

print('Select the Gemma results zip (results_gemma_*.zip):')
uploaded = files.upload()
for fname, data in uploaded.items():
    with open(f'/content/{fname}', 'wb') as f:
        f.write(data)
    with zipfile.ZipFile(f'/content/{fname}') as zf:
        zf.extractall('.')
    print(f'✓ Extracted {fname}')

import os
gemma_files = [f for f in os.listdir('results') if 'gemma' in f]
print(f'Gemma result files: {gemma_files}')

In [ ]:
import os
needed = [
    'results/detection_llama_summary.json',
    'results/detection_gemma_summary.json',
    'results/length_curves_llama.json',
    'results/length_curves_gemma.json',
    'results/robustness_llama.json',
    'results/robustness_gemma.json',
    'results/delta_sweep_llama.json',
    'results/delta_sweep_gemma.json',
]
all_present = True
for path in needed:
    exists = os.path.exists(path)
    status = '✓' if exists else '✗ MISSING'
    print(f'{status}  {path}')
    if not exists:
        all_present = False

if all_present:
    print('\n✅  All result files present — ready to generate figures')
else:
    print('\n⚠️  Some files missing — check which model notebook is still running')

---
## Step 2 — Generate figures
**~2 min.**

Output: `figures/fig{2-6}.pdf`

In [ ]:
!python scripts/make_figures.py

In [ ]:
from pathlib import Path
print('Generated figures:')
for f in sorted(Path('figures').glob('*.pdf')):
    print(f'  {f.name}  ({f.stat().st_size // 1024} KB)')

---
## Step 3 — Side-by-side summary table

Key numbers for the report.

In [ ]:
import json

rows = []
for fname, label in [
    ('results/detection_llama_summary.json', 'LLaMA 3.1 8B'),
    ('results/detection_gemma_summary.json', 'Gemma 2 9B'),
]:
    with open(fname) as f:
        s = json.load(f)
    long_key = next((k for k in s if 'ge150' in k), None)
    rows.append({
        'model':      label,
        'tpr_all':    s['tpr_at_1pct_fpr_all'],
        'tpr_150':    s.get(long_key, float('nan')),
        'ppl_ratio':  s['ppl_ratio_wm_over_uwm'],
        'threshold':  s['calibrated_z_threshold'],
    })

print(f"{'Model':<18} {'TPR(all)':>9} {'TPR(≥150t)':>11} {'PPL ratio':>10} {'z-thresh':>9}")
print('-' * 62)
for r in rows:
    print(f"{r['model']:<18} {r['tpr_all']:>9.1%} {r['tpr_150']:>11.1%} {r['ppl_ratio']:>10.3f} {r['threshold']:>9.3f}")

print()
print('Robustness (TPR @ 1% FPR under attack):')
rob_data = {}
for fname, label in [
    ('results/robustness_llama.json', 'LLaMA'),
    ('results/robustness_gemma.json', 'Gemma'),
]:
    with open(fname) as f:
        rob_data[label] = json.load(f)

all_conditions = list(rob_data['LLaMA'].keys())
print(f"{'Condition':<30} {'LLaMA':>8} {'Gemma':>8}")
print('-' * 50)
for cond in all_conditions:
    l = rob_data['LLaMA'].get(cond, {}).get('tpr_at_1pct_fpr', float('nan'))
    g = rob_data['Gemma'].get(cond, {}).get('tpr_at_1pct_fpr', float('nan'))
    print(f"{cond:<30} {l:>8.1%} {g:>8.1%}")

---
## Download figures

**Run before closing the session.**

In [ ]:
import zipfile
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
archive   = f'/content/figures_{timestamp}.zip'

with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in Path('figures').rglob('*'):
        if fp.is_file():
            zf.write(fp)

size_mb = Path(archive).stat().st_size / 1e6
print(f'Created {archive}  ({size_mb:.1f} MB)')

from google.colab import files
files.download(archive)